# End to end: slab → calculator → `get_potential_energy()` submits → visualise
The pipeline is ASE-native: attaching a VASP calculator and asking for the energy writes the inputs and runs `qsub` inside the calculation folder. The call returns once the job is queued.

## 1. Environment and resources
`jobscript.jobscript(...)` writes the PBS script and points `ASE_VASP_COMMAND` at it (same call as in the original notebooks).

In [ ]:
import gadivasp as gv
from gadivasp import jobscript, structures, plotting
from ase.build import bulk, molecule

cfg = gv.load_config()                       # project, scratch_root, modules (config.yaml)
jobscript.jobscript(ncpus=96, mem='100gb', walltime='4:00:00', cfg=cfg)

## 2. Slab creation (relaxed bulk → surface → supercell → sorted by z)

In [ ]:
ni = bulk('Ni', 'fcc', a=3.524, cubic=False)          # use your relaxed bulk here
slab = structures.slab_from_bulk(ni, (1, 1, 1), layers=5, vacuum=5.0, repeat=(4, 4))
structures.fix_indices(slab, range(len(slab) - 3 * 16))   # bottom two layers fixed
plotting.structure_views(slab, title='Ni(111), 5 layers');

## 3. Adsorbate

In [ ]:
ads = structures.adsorb_atop(slab, molecule('NH3'), height=2.0)   # N atop the top-layer Ni
plotting.structure_views(ads, title='NH3 on Ni(111)');

## 4. Calculator setter + submit
`Calculation(...)` attaches the ASE `Vasp` calculator using the presets (`relax`, `scf`, `zpe`, ...). `charge=+1` gives the 1e slab: `NELECT` is derived from the atoms. `submit()` is `atoms.get_potential_energy()`, guarded so it will not resubmit a job that is already queued or finished.

In [ ]:
calc = gv.Calculation('nh3_slab_1e', ads, cfg, preset='relax', charge=+1, dipole=True,
                      resources=dict(ncpus=96, mem='100gb', walltime='4:00:00'))
calc.write_inputs()      # optional: inspect INCAR / job.sh first
calc.submit()

## 5. Monitor

In [ ]:
calc.status()          # Queued / Running / converged
# gv.wait_for([calc.job_id])   # or block until it finishes

## 6. Results and visualisation

In [ ]:
print(calc.energy(), calc.converged())
plotting.structure_views(calc.atoms(), title='relaxed');

### The same thing in the original notebook idiom
```python
from ase.calculators.vasp import Vasp
jobscript.jobscript(ncpus=84, mem='60gb', walltime='4:00:00')   # sets ASE_VASP_COMMAND
calc = Vasp(directory='.../1e', **gv.make_params('relax', charge=1, neutral_nelect=800), atoms=slab)
slab.get_potential_energy()      # submits; FileNotFoundError(OUTCAR) just means 'queued'
```